# Fetch TCEQ Central Records by Proj Number

Source: https://records.tceq.texas.gov/

The home page redirects to the search service (`/cs/idcplg?IdcService=TCEQ_SEARCH`) and the search form is just a `GET` to `/cs/idcplg`. To pull the files for a project, query on `Secondary ID`, then download the documents we care about.

In [ ]:
import re
import time
from io import StringIO
from pathlib import Path
from urllib.parse import unquote, urljoin

import pandas as pd
import requests
from lxml import html

BASE_URL = 'https://records.tceq.texas.gov/'
ENDPOINT = BASE_URL + 'cs/idcplg'

# Create project root
def find_project_root(marker: str = '.git') -> Path:
    '''Climbs up folders starting from CWD until it finds the root.'''
    current = Path.cwd().resolve()

    # Loop upward until hitting the project root (where current == its own parent)
    while current != current.parent:
        if (current / marker).exists():
            return current
        current = current.parent

PROJECT_ROOT = find_project_root(marker='.git')
OUT_DIR = PROJECT_ROOT / 'data' / 'raw' / 'records'

In [2]:
DROP_COLS = ['Spacer Column', 'Select', 'Actions']

def _download_urls(page_html: str) -> list:
    '''
    Get the download url associated with each Content ID.

    Because the Download ID is different from the Content ID in the table, the link itself is grabbed straight from <a href> instead of reconstructing it ourselves.
    '''
    tbl = html.fromstring(page_html).get_element_by_id('table_0')
    rows = tbl.xpath('.//tr')

    # Retrieve column containing Content ID. Note row 0 contains the column names
    header = [' '.join(col.xpath('.//text()')).strip() for col in rows[0]]
    content_id = header.index('Content ID')

    urls = []
    for row in rows[1:]:
        hrefs = row[content_id].xpath('.//a/@href')
        urls.append(urljoin(BASE_URL, hrefs[0]) if hrefs else None)
    return urls


def fetch(secondary_id: int, session: requests.Session) -> pd.DataFrame:
    ''' 
    Fetch the data.

    All inputs, including blank ones, are required. The only one of interest is 'select0' and 'input0'.
    '''
    params = {
        'IdcService': 'TCEQ_PERFORM_SEARCH',
        'xIdcProfile': 'Record',
        'IsExternalSearch': '1',
        'sortSearch': 'false',
        'newSearch': 'true',
        'xRecordSeries': '0',
        'xInsightDocumentType': '0',
        'xMedia': '0',
        'operator': 'AND',
        'select0': 'xSecondaryID', 'input0': str(secondary_id),             
        'select1': '', 'input1': '',
        'select2': '', 'input2': '',
        'select3': '', 'input3': '',
        'ftx': '',
    }
    resp = session.get(ENDPOINT, params=params, timeout=120)
    resp.raise_for_status()

    try:
        raw = pd.read_html(StringIO(resp.text), attrs={'id': 'table_0'})[0]
    except ValueError:
        return pd.DataFrame()

    raw.columns = raw.iloc[0]                   # First row is the column names
    df = raw.iloc[1:].drop(columns=DROP_COLS)
    df['Download URL'] = _download_urls(resp.text)
    return df

## Download a record's file

In [4]:
def _filename_from_response(resp: requests.Response) -> str:
    '''
    Constructs filename for the file using the Content-Disposition header when downloading.
    '''
    content_dis = resp.headers.get('Content-Disposition', '')
    match = re.search(r"filename\*=UTF-8''([^;]+)", content_dis) or re.search(r'filename="?([^";]+)', content_dis)
    
    # Clean the filename or return a fallback one
    if match:
        return unquote(match.group(1)).strip()
    return f"record-{resp.url.split('dID=')[-1].split('&')[0]}.pdf"


def download_file(url: str, session: requests.Session, out_dir: Path = OUT_DIR) -> Path:
    ''' 
    Downloads file associated with the Content ID.
    '''
    resp = session.get(url, timeout=180)
    resp.raise_for_status()

    out_dir.mkdir(parents=True, exist_ok=True)
    dest = out_dir / _filename_from_response(resp)
    dest.write_bytes(resp.content)

    return dest

## Extract Relevant Proj Num

In [5]:
CANDIDATES_CSV = PROJECT_ROOT / 'data' / 'processed' / 'candidates-list.csv'

# Project Number is the same value the records search calls the Secondary ID
def load_project_numbers(csv_path: Path = CANDIDATES_CSV) -> list:
    projects = pd.read_csv(csv_path, usecols=['Project Number'])['Project Number']
    return projects.astype(int).unique().tolist()

CANDIDATE_IDS = load_project_numbers()
print(f'{len(CANDIDATE_IDS)} project numbers')

31 project numbers


## Configure & run

Loop the project numbers and, for each, grab the record whose Title is
`DOWNLOAD_TITLE` (e.g. Technical Review) into `data/raw/records/`. The results
table is only there to find the download link, we don't keep it.

In [ ]:
# Change values here for desired scope
SECONDARY_IDS = CANDIDATE_IDS
DOWNLOAD_TITLE = 'Technical Review'   # set to None to skip the downloads
SLEEP_SECONDS = 1.0

OUT_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers['User-Agent'] = 'Mozilla/5.0'

for secondary_id in SECONDARY_IDS:
    df = fetch(secondary_id, session)
    print(f'Secondary ID {secondary_id} -> {len(df):>4} rows')

    if DOWNLOAD_TITLE and not df.empty:
        for url in df.loc[df['Title'] == DOWNLOAD_TITLE, 'Download URL']:
            saved = download_file(url, session)
            print(f'    downloaded  {saved.relative_to(PROJECT_ROOT)}')
    time.sleep(SLEEP_SECONDS)

Secondary ID 408828 ->    6 rows
    downloaded  data/raw/records/AIR NSR_180360-408828_Permits_Public_20260520_Technical Review_8426413_.pdf
Secondary ID 381658 ->    6 rows
    downloaded  data/raw/records/AIR NSR_177873-381658_Permits_Public_20241016_Technical Review_7335946_.pdf
Secondary ID 396366 ->   20 rows
    downloaded  data/raw/records/AIR NSR_AirPermits-396366_Permits_Public_20260121_Technical Review_8201547_.pdf
Secondary ID 389727 ->    4 rows
    downloaded  data/raw/records/AIR NSR_179308-389727_Permits_Public_20250305_Technical Review_7630740_.pdf
Secondary ID 381183 ->    5 rows
    downloaded  data/raw/records/AIR NSR_177154-381183_Permits_Public_20241030_Technical Review_7365380_.pdf
Secondary ID 409617 ->    7 rows
    downloaded  data/raw/records/AIR NSR_180362-409617_Permits_Public_20260610_Technical Review_8492765_.pdf
Secondary ID 402447 ->    5 rows
    downloaded  data/raw/records/AIR NSR_182541-402447_Permits_Public_20260108_Technical Review_8180573_.pdf
Se